# Refresh: Productivity-Adjusted Wage Measures

This compact notebook refreshes the chart from the long services-ex-housing wage VAR notebook: **Productivity-adjusted wage measures, non-housing services**.

Change `START_DATE` in the first code cell, then run all cells. By default the notebook tries to download fresh FRED and Atlanta Fed data. If a download fails, it falls back to cached files in `cache/` when available.

Outputs are saved to `output/services_ex_housing_var_adjusted/figures/chicago_style_productivity_adjusted_wage_comparison.png` and `output/services_ex_housing_var_adjusted/quarterly_productivity_adjusted_wage_measures.csv`.

In [ ]:
from pathlib import Path
from urllib.request import Request, urlopen
import numpy as np
import pandas as pd

# User settings
START_DATE = "2014-01-01"  # Change this to move the first date shown in the chart.
DATA_START_DATE = "2005-01-01"  # Keep several years before START_DATE for growth calculations.
REFRESH_DATA = True  # Set False to use cached data only.

ROOT = Path(".").resolve()
CACHE = ROOT / "cache"
OUT = ROOT / "output" / "services_ex_housing_var_adjusted"
FIG = OUT / "figures"
CACHE.mkdir(exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

ATLANTA_WGT_URL = (
    "https://www.atlantafed.org/-/media/Project/Atlanta/FRBA/Documents/"
    "datafiles/chcs/wage-growth-tracker/wage-growth-data.xlsx"
)

SERIES = {
    "private_services_va_nominal": "VASPI",
    "private_services_va_deflator": "VAPISPI",
    "housing_va_nominal": "A2009C1Q027SBEA",
    "housing_va_deflator": "A2009G3Q086SBEA",
    "private_services_hours": "CES0800000034",
    "eci_comp_services": "CIS201S000000000I",
    "eci_wages_services": "CIS202S000000000I",
}

CHART_COLUMNS = [
    "eci_comp_ulc_growth_4q",
    "eci_wages_ulc_growth_4q",
    "atlanta_wgt_ulc_growth_4q",
]

CHART_LABELS = {
    "eci_comp_ulc_growth_4q": "ECI compensation ULC",
    "eci_wages_ulc_growth_4q": "ECI wages ULC",
    "atlanta_wgt_ulc_growth_4q": "Atlanta WGT minus productivity",
}

CHART_COLORS = {
    "eci_comp_ulc_growth_4q": "#2563eb",
    "eci_wages_ulc_growth_4q": "#0f766e",
    "atlanta_wgt_ulc_growth_4q": "#7c3aed",
}


In [ ]:
def read_cached_fred(series_id):
    csv_path = CACHE / f"fred_{series_id}.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path, parse_dates=["date"])
        return df.set_index("date")["value"].sort_index().rename(series_id)

    legacy_pickle = CACHE / f"fred_csv_{series_id}.pkl"
    if legacy_pickle.exists():
        return pd.read_pickle(legacy_pickle).sort_index().rename(series_id)

    return None


def write_cached_fred(series_id, series):
    path = CACHE / f"fred_{series_id}.csv"
    series.rename("value").to_frame().rename_axis("date").to_csv(path)


def download_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    df = pd.read_csv(url)
    df = df.rename(columns={df.columns[0]: "date", df.columns[1]: "value"})
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    series = df.set_index("date")["value"].dropna().sort_index().rename(series_id)
    write_cached_fred(series_id, series)
    return series


def fred_series(series_id):
    if not REFRESH_DATA:
        cached = read_cached_fred(series_id)
        if cached is not None:
            return cached

    try:
        return download_fred(series_id)
    except Exception as exc:
        cached = read_cached_fred(series_id)
        if cached is not None:
            print(f"Using cached FRED data for {series_id}; download failed: {exc}")
            return cached
        raise


def quarterly_fred_level(series_id, name):
    return fred_series(series_id).rename(name).resample("QS").last()


def quarterly_average_monthly_fred(series_id, name):
    return fred_series(series_id).rename(name).resample("QS").mean()


def load_atlanta_wgt():
    xlsx_path = CACHE / "wgt.xlsx"
    csv_path = CACHE / "wgt_industry.csv"

    if REFRESH_DATA or not xlsx_path.exists():
        try:
            req = Request(ATLANTA_WGT_URL, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(req, timeout=120) as response:
                xlsx_path.write_bytes(response.read())
        except Exception as exc:
            if not xlsx_path.exists() and not csv_path.exists():
                raise
            print(f"Using cached Atlanta Fed WGT data; download failed: {exc}")

    if xlsx_path.exists():
        df = pd.read_excel(xlsx_path, sheet_name="Industry", header=2)
        df = df.rename(columns={df.columns[0]: "date"})
        df["date"] = pd.to_datetime(df["date"])
        for col in df.columns[1:]:
            df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df.set_index("date").sort_index()
        df.to_csv(csv_path)
        return df

    return pd.read_csv(csv_path, parse_dates=["date"]).set_index("date").sort_index()


def log_growth(series, periods=1, annualize=1):
    return 100 * annualize * np.log(series / series.shift(periods))


In [ ]:
def build_productivity_adjusted_wage_panel():
    raw = pd.concat(
        [
            quarterly_fred_level(SERIES["private_services_va_nominal"], "private_services_va_nominal"),
            quarterly_fred_level(SERIES["private_services_va_deflator"], "private_services_va_deflator"),
            quarterly_fred_level(SERIES["housing_va_nominal"], "housing_va_nominal"),
            quarterly_fred_level(SERIES["housing_va_deflator"], "housing_va_deflator"),
            quarterly_average_monthly_fred(SERIES["private_services_hours"], "private_services_hours"),
            quarterly_fred_level(SERIES["eci_comp_services"], "eci_comp_services"),
            quarterly_fred_level(SERIES["eci_wages_services"], "eci_wages_services"),
            load_atlanta_wgt()["Overall"].resample("QS").mean().rename("atlanta_wgt_overall"),
        ],
        axis=1,
    ).loc[pd.Timestamp(DATA_START_DATE):].sort_index()

    raw["nonhousing_services_va_nominal"] = (
        raw["private_services_va_nominal"] - raw["housing_va_nominal"]
    )
    raw["housing_va_share_lag"] = (
        raw["housing_va_nominal"] / raw["private_services_va_nominal"]
    ).shift(1)

    services_deflator_log_change = np.log(raw["private_services_va_deflator"]).diff()
    housing_deflator_log_change = np.log(raw["housing_va_deflator"]).diff()
    nonhousing_deflator_log_change = (
        services_deflator_log_change
        - raw["housing_va_share_lag"] * housing_deflator_log_change
    ) / (1 - raw["housing_va_share_lag"])

    panel = raw.copy()
    panel["nonhousing_services_va_deflator_4q"] = 100 * nonhousing_deflator_log_change.rolling(4).sum()
    panel["nonhousing_services_nominal_va_growth_4q"] = log_growth(
        panel["nonhousing_services_va_nominal"], periods=4
    )
    panel["private_services_hours_growth_4q"] = log_growth(
        panel["private_services_hours"], periods=4
    )
    panel["nonhousing_services_real_va_growth_4q"] = (
        panel["nonhousing_services_nominal_va_growth_4q"]
        - panel["nonhousing_services_va_deflator_4q"]
    )
    panel["nonhousing_services_productivity_growth_4q"] = (
        panel["nonhousing_services_real_va_growth_4q"]
        - panel["private_services_hours_growth_4q"]
    )

    for prefix in ["eci_comp", "eci_wages"]:
        panel[f"{prefix}_growth_4q"] = log_growth(panel[f"{prefix}_services"], periods=4)
        panel[f"{prefix}_ulc_growth_4q"] = (
            panel[f"{prefix}_growth_4q"]
            - panel["nonhousing_services_productivity_growth_4q"]
        )

    panel["atlanta_wgt_ulc_growth_4q"] = (
        panel["atlanta_wgt_overall"]
        - panel["nonhousing_services_productivity_growth_4q"]
    )

    panel.index.name = "date"
    return panel


panel = build_productivity_adjusted_wage_panel()
panel.to_csv(OUT / "quarterly_productivity_adjusted_wage_measures.csv")

print("Available chart sample:", panel[CHART_COLUMNS].dropna().index.min().date(), "to", panel[CHART_COLUMNS].dropna().index.max().date())
panel[CHART_COLUMNS + ["nonhousing_services_productivity_growth_4q"]].tail().round(3)


In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise ImportError("matplotlib is required for the chart cell. Install it in the notebook kernel, then rerun this cell.") from exc


def plot_productivity_adjusted_wages(panel, start_date=START_DATE):
    d = panel.loc[pd.Timestamp(start_date):, CHART_COLUMNS].dropna()
    if d.empty:
        raise ValueError(f"No chart data are available on or after {start_date}. Try an earlier START_DATE.")

    fig, ax = plt.subplots(figsize=(9.2, 4.6))
    ax.axhline(0, color="#9ca3af", lw=1)
    for col in CHART_COLUMNS:
        ax.plot(d.index, d[col], color=CHART_COLORS[col], lw=1.9, label=CHART_LABELS[col])

    ax.set_title("Productivity-adjusted wage measures, non-housing services")
    ax.set_ylabel("Four-quarter percent change")
    ax.grid(axis="y", color="#e5e7eb", lw=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()

    path = FIG / "chicago_style_productivity_adjusted_wage_comparison.png"
    fig.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path


chart_path = plot_productivity_adjusted_wages(panel)
print(f"Saved chart to: {chart_path}")
